In [ ]:
#instalar libreria wfdb para trabajr con PhysioNet
# Instalar una versión de pandas compatible y wfdb
!pip install "pandas<2.2.0" wfdb

##!pip install wfdb

# ANOTACIONES


**"|" → Separación entre registros**

  * Significa que empieza o termina un segmento dentro del registro.
  * En la práctica, en MIT-BIH se usa poco, pero puede indicar que hay “cortes” o que una parte del ECG no se registró.


**"+" → Información adicional (metadatos)**

* Marca donde empieza una anotación de parámetros: ganancia de la señal, nombre del paciente, calibración, etc.
* No es un latido, es como un “marcador administrativo”.


**"~" → Ruido o artefacto**

* Indica un tramo donde la señal está contaminada (ej. movimiento del paciente, electrodos mal colocados).
* Es útil porque para excluir esos trozos en el análisis, ya que confunden a los algoritmos de detección.

**"N" → Normal**

**"V" → Ventricular prematuro**
* **"L"**	Left bundle branch block beat (LBBB)	Bloqueo de rama izquierda
* **"R"**	Right bundle branch block beat (RBBB)	Bloqueo de rama derecha
* **"B"**	Bundle branch block beat (unspecified)	Bloqueo de rama no especificado

**"A" → Auricular prematuro**
* **"a"**	Aberrated atrial premature beat	// Variante “aberrante” de APC
* **"J"**	Nodal (junctional) premature beat	// Latido prematuro desde nodo AV
* **"S"**	Supraventricular premature beat	// Prematuro de origen auricular o nodal
* **"V"**	Premature ventricular contraction (PVC) //	Latido prematuro desde ventrículo
* **"E"**	Ventricular escape beat //	Latido “de escape” desde ventrículo

**Fusión**
* **"F"** Fusion of ventricular and normal beat //
Mezcla entre un PVC y un latido normal
* **"f"** Fusion of paced and normal beat
// Mezcla de latido normal y marcapasos


#PREPARACIÓN

In [ ]:
#librerias
import wfdb
import matplotlib.pyplot as plt
import math
import numpy as np

In [ ]:
#descargar
##leer archivos .dat y .hea, los nombes son 100.dat y 100.hea
record100 = wfdb.rdrecord('100', pn_dir='mitdb')
record101 = wfdb.rdrecord('101', pn_dir='mitdb')
record102 = wfdb.rdrecord('102', pn_dir='mitdb')

##leer anotaciones .atr
anotacion100 = wfdb.rdann('100', 'atr', pn_dir='mitdb')
anotacion101 = wfdb.rdann('101', 'atr', pn_dir='mitdb')
anotacion102 = wfdb.rdann('102', 'atr', pn_dir='mitdb')

#visualizar
wfdb.plot_wfdb(record=record100, annotation=anotacion100, title='MIT-BIH Arrhythmia Record 100')

wfdb.plot_wfdb(record=record101, annotation=anotacion101, title='MIT-BIH Arrhythmia Record 101')

In [ ]:
# Definir un rango de muestras
fs100 = record100.fs
start = int(5 * fs100)   # desde 5 segs
end   = int(15 * fs100)  # hasta 15 segs

# Extraer señal en ese rango
signal = record100.p_signal[start:end, 0]
time = [t/fs100 for t in range(start, end)]

# Dibujar señal
plt.figure(figsize=(12,4))
plt.plot(time, signal, label="ECG (MLII)")
plt.xlabel("Tiempo (s)")
plt.ylabel("mV")

# Añadir anotaciones dentro de ese rango
for sample, symbol in zip(anotacion100.sample, anotacion100.symbol):
    if start <= sample < end:
        plt.axvline(sample/fs100, color="red", linestyle="--", alpha=0.7)
        plt.text(sample/fs100, max(signal), symbol, color="red")

plt.title("MIT-BIH Record 100 (zoom 5-15s)")
plt.legend()
plt.show()

#MUESTREO

In [ ]:
#PARA PODER VER ondas P, QRS y T

# Definir un rango de muestras
start = int(4 * fs100)   # desde 1515 segundos
end   = int(9 * fs100)  # hasta 1525 segundos

# Extraer señal en ese rango
signal = record100.p_signal[start:end, 0]  # canal 0 (MLII)
time = [t/fs100 for t in range(start, end)]

# Dibujar señal
plt.figure(figsize=(12,4))
plt.plot(time, signal, label="ECG (MLII)")
plt.xlabel("Tiempo (s)")
plt.ylabel("mV")

# Añadir anotaciones dentro de ese rango
for sample, symbol in zip(anotacion100.sample, anotacion100.symbol):
    if start <= sample < end:
        plt.axvline(sample/fs100, color="red", linestyle="--", alpha=0.7)
        plt.text(sample/fs100, max(signal), symbol, color="red")

plt.title("MIT-BIH Record 100")
plt.legend()
plt.show()


In [ ]:

#PARA PODER VER ondas P, QRS y T

# Definir un rango de muestras
fs101 = record101.fs
start = int(318 * fs101)
end   = int(323 * fs101)

# Extraer señal en ese rango
signal = record101.p_signal[start:end, 0]  # canal 0 (MLII)
time = [t/fs101 for t in range(start, end)]

# Dibujar señal
plt.figure(figsize=(12,4))
plt.plot(time, signal, label="ECG (MLII)")
plt.xlabel("Tiempo (s)")
plt.ylabel("mV")

# Añadir anotaciones dentro de ese rango
for sample, symbol in zip(anotacion101.sample, anotacion101.symbol):
    if start <= sample < end:
        plt.axvline(sample/fs101, color="red", linestyle="--", alpha=0.7)
        plt.text(sample/fs101, max(signal), symbol, color="red")

plt.title("MIT-BIH Record 101")
plt.legend()
plt.show()

In [ ]:
print("Frecuencia de muestreo 100:", fs100)
print("Frecuencia de muestreo 101:", fs101)
print("Frecuencia de muestreo 101:", fs102)
print("Duración (muestras):", len(record100.p_signal))
print("Anotaciones disponibles 100:", set(anotacion100.symbol))
print("Anotaciones disponibles 101:", set(anotacion101.symbol))
print("Anotaciones disponibles 101:", set(anotacion102.symbol))

In [ ]:
# Filtrar latidos que no sean N
# no_normales => lista de tuplas (muestra,sym)
no_normales = [(anotacion101.sample[i], anotacion101.symbol[i])
               for i, sym in enumerate(anotacion101.symbol) if sym != 'N'] #enumerate() devuelve pares (muestra, sym).

no_normales_tiempo = [(pos, sym, pos/fs101) for pos, sym in no_normales]

print("Total no normales 101:", len(no_normales), "\n")

# Imprimir lista solo simbolo y tiempo
for _, sym, t in no_normales_tiempo:
    print(f"{sym} → {t:.0f}")

In [ ]:
no_normales = [(anotacion100.sample[i], anotacion100.symbol[i])
               for i, sym in enumerate(anotacion100.symbol) if sym != 'N']

no_normales_tiempo = [(pos, sym, pos/fs100) for pos, sym in no_normales[:20]]

print("Total no normales 100:", len(no_normales), "\n")

# Imprimir lista solo simbolo y tiempo
for _, sym, t in no_normales_tiempo:
    print(f"{sym} → {t:.0f}")

In [ ]:
fs102 = record102.fs
no_normales = [(anotacion102.sample[i], anotacion102.symbol[i])
               for i, sym in enumerate(anotacion102.symbol) if sym != 'N']

no_normales_tiempo = [(pos, sym, pos/fs100) for pos, sym in no_normales[:20]]

print("Total no normales 102:", len(no_normales), "\n")

# Imprimir lista solo simbolo y tiempo
for _, sym, t in no_normales_tiempo:
    print(f"{sym} → {t:.0f}")

In [ ]:
# Filtrar latidos que no sean N
# no_normales => lista de tuplas (muestra,sym)
no_normales=[];
no_normales_tiempo=[];
no_normales = [(anotacion100.sample[i], anotacion100.symbol[i])
               for i, sym in enumerate(anotacion100.symbol) if sym != 'N'] #enumerate() devuelve pares (muestra, sym).

no_normales_tiempo = [(pos, sym, pos/fs100) for pos, sym in no_normales]

print("Total no normales 100:", len(no_normales), "\n")

# Imprimir lista solo simbolo y tiempo
for _, sym, t in no_normales_tiempo:
    print(f"{sym} → {t:.0f}")

Todos los datos del record 100 son válidos!

#2.2

In [ ]:
#aplicar FSM sobre un registro MIT-BIH y guarda índices QRS detectados.

import wfdb, numpy as np, pandas as pd, os

# ---- parámetros ----
MA_TAPS = 30            # ventana media móvil (integración)
THRESH_FACTOR = 1.5     # umbral = mean + factor * std
WAIT_SAMPLES = int(0.20 * 360)    # espera ≈200 ms
REFRACT_SAMPLES = int(0.20 * 360) # refractario ≈200 ms
OUTDIR = "detecciones_minimas"
os.makedirs(OUTDIR, exist_ok=True)

# ---- leer señal ----
sig = record100.p_signal[:, 0].astype(float) #usando todas las muestras del canal 0 (MLII)
N = len(sig)

# ---- preprocesado mínimo (derivada -> abs -> media móvil) ----
derivada = np.diff(sig, prepend=sig[0])
rect = np.abs(derivada) #val absoluto pq sino se pueden cancelar
smooth = np.convolve(rect, np.ones(MA_TAPS)/MA_TAPS, mode='same')

# ---- umbral ----
threshold = np.mean(smooth) + THRESH_FACTOR * np.std(smooth)

# ---- FSM (mínima: LOOKING, PROV, HALF) ----
LOOKING, PROV, HALF = 1, 2, 3
state = LOOKING
cand_val = -1.0
cand_idx = -1
prov_best_idx = -1
prov_best_val = -1.0
wait_until = -1
last_confirmed = -100000
detected = []

for i in range(N):
    v = smooth[i]
    if state == LOOKING:
        if v > threshold:
            state = PROV
            cand_val = v
            cand_idx = i
    elif state == PROV:
        if v > cand_val:
            cand_val = v
            cand_idx = i
        if v < 0.5 * cand_val:
            state = HALF
            prov_best_idx = cand_idx
            prov_best_val = cand_val
            wait_until = i + WAIT_SAMPLES
    elif state == HALF:
        if v > prov_best_val:
            state = PROV
            cand_val = v
            cand_idx = i
        else:
            if i >= wait_until:
                if prov_best_idx - last_confirmed > REFRACT_SAMPLES:
                    detected.append(int(prov_best_idx))
                    last_confirmed = prov_best_idx
                state = LOOKING
                cand_val = -1.0
                cand_idx = -1
                prov_best_idx = -1
                prov_best_val = -1.0
                wait_until = -1

# ---- guardar y mostrar ----
detected = np.array(detected, dtype=int)
print(f"Total de picos detectados en el registro 100: {len(detected)}")
print(f"Registro 100: detectados {len(detected)} picos QRS (primeros 20): {detected[21:50]}")

Total de picos detectados en el registro 100: 2272
Registro 100: detectados 2272 picos QRS (primeros 20): [ 6214  6527  6820  7101  7387  7665  7949  8241  8535  8837  9142  9428
  9708  9994 10278 10587 10894 11187 11476 11776 12062 12347 12643 12947
 13261 13559 13841 14128 14419]


#**2.2 EN "TIEMPO REAL"**

In [ ]:
# ---------- Parámetros ----------
THRESHOLD = 0.3
WAIT_SAMPLES = 72 #200 ms a 360MHz
FS = 360
REFRACT_SAMPLES = 72
#no puede haber dos QRS muy seguidos
#dos picos casi jutnos en realidad forman parte del mismo latido
#dejamos un espacio llamado REFRACTORIO de mínimo 200ms en 360MHz => 72 samples

sig = record100.p_signal[:, 0].astype(float) #usando todas las muestras del canal 0 (MLII)

# ---------- Leer señal ----------
signal = record100.p_signal[:,0]   # canal 0
N = len(signal) #POR AHORA PARA PONER UN FINAL


# ---------- Inicializar variables ----------
state = "RESET"
last_confirmed = -100000
detected = []
k=3

# ---------- Bucle principal ----------
for i in range(k, N):
    # 1) derivada simple
    d = signal[i] - signal[i-k]
    v = abs(d)   # valor absoluto
    idx = i-5


    # ---------- Máquina de estados ----------
    if state == "RESET":
        # reiniciar
        state = "LOOKING"
        cand_val = -1.0 # valor candidato a QRS
        cand_idx = -1 # pos del candidato a QRS
        prov_best_idx = -1 # valor def QRS
        prov_best_val = -1.0 #pos de def QRS
        wait_until = -1


    if state == "LOOKING":
        if v > THRESHOLD: #si hay subida notable
            state = "PROV"
            cand_val = v
            cand_idx = idx


    elif state == "PROV":
        if v > cand_val: #si val > max anterior
            cand_val = v
            cand_idx = idx
        # si baja a menos de la mitad del pico provisional -> pasamos a HALF
        if v < 0.5 * cand_val:
            state = "HALF"
            prov_best_val = cand_val
            prov_best_idx = cand_idx
            wait_until = prov_best_idx + WAIT_SAMPLES #ESPERAMOS +72 samples


    elif state == "HALF":
        if v > prov_best_val: #si aparece v mayor que max hasta ahora
            state = "PROV" #volvemos a PROV
            cand_val = v
            cand_idx = idx
        else:
            # si esperamos suficiente, confirmamos el pico
            if i >= wait_until:
                if prov_best_idx - last_confirmed > REFRACT_SAMPLES:
                  #si no cumple el REFRACTORIO no se confirma pico
                    detected.append(int(prov_best_idx))
                    last_confirmed = prov_best_idx #pos del ultimo pico detectado
                # reiniciar
                state = "RESET"


# ---------- Mostrar resultados ----------
print("Total de picos detectados:", len(detected))


# ---------- Gráfica ----------
t = np.arange(len(sig)) / FS


plt.figure(figsize=(12,4))
plt.plot(t, sig, linewidth=0.8, label='ECG (canal 0)')
if len(detected):
    plt.scatter(np.array(detected)/FS, np.array(signal)[detected],
                color='red',label='Picos detectados')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud ECG')
plt.title('ECG y picos detectados')
X=1
plt.xlim(X, min(X+0.1, len(sig)/FS))   # muestra hasta 10 s por defecto
plt.legend()
plt.show()

print("Índices de picos detectados:", detected)

In [ ]:
# 20 primeras posiciones (índices de muestra) de las anotaciones del registro 100
print(f"Total de anotaciones en el registro 100: {len(anotacion100.symbol)}")
print("Primeras 20 posiciones de anotaciones:", anotacion100.sample[0:50])

#**2.3 HERMITE**

#Teoría

##Entender polinomio de Hermite

Los polinomios de Hermite son simplemente una colección de "formas prefabricadas", como moldes con formas curvas.
- Forma 0 ($H_0$): Es una campana de Gauss. Se parece mucho a un latido QRS normal.
- Forma 1 ($H_1$): Es como una "S" (sube y baja). Sirve para describir si el latido está inclinado.
- Forma 2 ($H_2$): Tiene forma de "W" (baja, sube, baja). Sirve para describir si el latido es más estrecho o tiene rebotes.


In [ ]:
import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt
import math

def hermite_functions(t, n, sigma):
    # La fórmula matemática para crear las formas
    norm = 1.0 / np.sqrt(sigma * (2**n) * math.factorial(n) * np.sqrt(np.pi))
    gauss = np.exp(-t**2 / (2 * sigma**2))
    Hn = sp.hermite(n)(t / sigma)
    return norm * gauss * Hn

# Vamos a pintar las 4 primeras formas
t = np.arange(-40, 40, 0.1) # Eje de tiempo
sigma = 10

plt.figure(figsize=(10, 6))

# Forma 0: La montaña (Se parece al QRS)
plt.plot(t, hermite_functions(t, 0, sigma), label='Orden 0 (La base)', linewidth=3)

# Forma 1: La inclinación
plt.plot(t, hermite_functions(t, 1, sigma), label='Orden 1 (Inclinación)')

# Forma 2: Los detalles
plt.plot(t, hermite_functions(t, 2, sigma), label='Orden 2 (Detalles)')

plt.title("Molde (Polinomios de Hermite)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 1. Calculamos las piezas
base = hermite_functions(t, 0, sigma)   # Azul (Montaña)
inclinador = hermite_functions(t, 1, sigma) # Naranja (S)
adelgazador = hermite_functions(t, 2, sigma) # Verde (W)

plt.figure(figsize=(12, 4))

# GRAFICA 1: La base sola
plt.subplot(1, 5, 1)
plt.plot(t, base, linewidth=3, color='blue')
plt.title("Solo la Base ($H_0$)\n(Latido Perfecto)")
plt.ylim(-0.1, 0.35)
plt.grid()

# GRAFICA 2: Base + Inclinador
# Le sumamos un poco de la curva naranja
latido_inclinadoD = base + (0.5 * inclinador)
plt.subplot(1, 5, 2)
plt.plot(t, base, '--', color='blue', alpha=0.3, label='Base')
plt.plot(t, latido_inclinadoD, linewidth=3, color='orange', label='Base + $H_1$')
plt.title("Base + $H_1$\n(Se desplaza a la dcha)")
plt.ylim(-0.1, 0.35)
plt.legend()
plt.grid()

# GRAFICA 2.1: Base - Inclinador
# Le sumamos un poco de la curva naranja
latido_inclinadoI = base - (0.5 * inclinador)
plt.subplot(1, 5, 3)
plt.plot(t, base, '--', color='blue', alpha=0.3, label='Base')
plt.plot(t, latido_inclinadoI, linewidth=3, color='orange', label='Base - $H_1$')
plt.title("Base - $H_1$\n(Se desplaza a la izq)")
plt.ylim(-0.1, 0.35)
plt.legend()
plt.grid()

# GRAFICA 3: Base - Adelgazador
# Le restamos la curva verde
latido_flaco = base - (0.5 * adelgazador)
plt.subplot(1, 5, 4)
plt.plot(t, base, '--', color='blue', alpha=0.3, label='Base')
plt.plot(t, latido_flaco, linewidth=3, color='green', label='Base - $H_2$')
plt.title("Base - $H_2$\n(Se hace más delgado)")
plt.ylim(-0.1, 0.35)
plt.legend(loc='lower left')
plt.grid()

# GRAFICA 4: Base + Adelgazador
# Le restamos la curva verde
latido_gordo = base + (0.5 * adelgazador)
plt.subplot(1, 5, 5)
plt.plot(t, base, '--', color='blue', alpha=0.3, label='Base')
plt.plot(t, latido_gordo, linewidth=3, color='green', label='Base + $H_2$')
plt.title("Base + $H_2$\n(Se hace más ancho)")
plt.ylim(-0.1, 0.35)
plt.legend(loc='lower left')
plt.grid()

plt.tight_layout()
plt.show()

1. Naranja ($H_1$): Es negativa a la izquierda y positiva a la derecha.
- Si la SUMAS: Le restas altura a la izquierda y se la das a la derecha $\rightarrow$ El pico se mueve a la derecha.
- Si la RESTAS: Haces lo contrario $\rightarrow$ El pico se mueve a la izquierda.
2. Verde ($H_2$): Tiene un "valle" (es negativa) justo en el centro.
- Si la SUMAS: Estás sumando un valle al pico de la montaña $\rightarrow$ La montaña baja de altura y se "desparrama" (más ancha/bajita).
- Si la RESTAS: Restar un valle es sumar un pico extra $\rightarrow$ La montaña crece y se afila (más alta/delgada).

**Conclusión**: da igual hacia qué lado se mueva o si engorda o adelgaza. Lo importante es que con los coefs, se puede dibujar exactamente la forma de un latido sin haber visto la señal original".

El 1.0 dice que hay un latido.

El +0.5 dice que está inclinado a la derecha.

El -0.2 dice que es más estrecho de lo normal.

In [ ]:
# 2. Configuración
NUM_COEFFS = 6    # Usaremos 6 números para describir cada latido
SIGMA = 5.0       # Anchura "estándar" del latido
WINDOW = 36       # Cogeremos 36 muestras a cada lado del pico (72/2)

mis_coeficientes = []
latidos_reales = []

print(f"Analizando {len(detected)} latidos detectados...")

for pico_idx in detected:
    # A) Recortar el latido (La ventana de tiempo)
    inicio = int(pico_idx - WINDOW)
    fin = int(pico_idx + WINDOW)

    # Seguridad: si el latido está muy al borde, lo saltamos
    if inicio < 0 or fin >= len(signal):
        continue

    # Extraemos el trozo de señal
    latido = signal[inicio:fin]
    # Restar media para que esté centrado en 0 (quitar baseline)
    latido = latido - np.mean(latido)
    latidos_reales.append(latido)

    # B) Calcular los coeficientes
    # Eje de tiempo centrado en 0 (-36 a +36)
    t_local = np.arange(-WINDOW, WINDOW)

    coeffs_latido = []

    # Probamos con los 6 coefs
    for n in range(NUM_COEFFS):
        molde = hermite_functions(t_local, n, SIGMA)

        # PROYECCIÓN: Latido * Molde y sumar
        peso = np.dot(latido, molde)
        coeffs_latido.append(peso)

    mis_coeficientes.append(coeffs_latido)

print("Análisis terminado!")


#---------------- VISUALIZACIÓN ----------------
if len(latidos_reales) > 0:
    # Elegimos el primer latido para ver si la reconstrucción es fiel
    idx_prueba = 0

    original = latidos_reales[idx_prueba]
    adn_latido = mis_coeficientes[idx_prueba] # Estos son los 6 coefs

    # Intentamos reconstruir el latido usando SOLO los 6 coefs
    t_local = np.arange(-WINDOW, WINDOW)
    reconstruccion = np.zeros_like(t_local, dtype=float)

    for n in range(NUM_COEFFS):
        molde = hermite_functions(t_local, n, SIGMA)
        reconstruccion += adn_latido[n] * molde # Sumamos cada molde con su peso

    # Pintamos
    plt.figure(figsize=(10, 5))
    plt.plot(t_local, original, label='Tu Latido Real', color='blue', linewidth=2)
    plt.plot(t_local, reconstruccion, '--', label='Reconstrucción (Usando solo 6 nums)', color='red', linewidth=2)
    plt.title(f"Prueba de Compresión Hermite (Sigma={SIGMA})")
    plt.legend()
    plt.grid(True)
    plt.show()

    print("ADN del latido (coefs):", np.round(adn_latido, 2))

In [ ]:
X = 15
ANCHO = 2
# -----------------

# Eje de tiempo
t = np.arange(len(signal)) / FS

plt.figure(figsize=(14, 6))

# 1. señal REAL (bd)
plt.plot(t, signal, color='steelblue', alpha=0.6, label='ECG Real (Database)')

# 2. línea ROJA (reconstrucción)
plt.plot(t, linea_roja_hermite, color='red', linestyle='--', linewidth=2, label='Reconstrucción Hermite')

plt.xlim(X, X + ANCHO)
plt.ylim(-0.6, 1)

plt.title(f"ECG Real vs Matemático (Segundos {X} a {X+ANCHO})")
plt.xlabel("Tiempo (s)")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()

#Código

In [ ]:
def hermite(t, n, sigma):
    x = t / sigma

    # --- 1. CALCULO DEL POLINOMIO Hn(x) ---
    # Usamos la relación: H_n = 2x*H_{n-1} - 2(n-1)*H_{n-2}

    if n == 0:
        Hn = 1.0
    elif n == 1:
        Hn = 2.0 * x
    else:
        h_n_2 = 1.0       # H0
        h_n_1 = 2.0 * x        # H1
        Hn = 0.0

        # Bucle desde 2 hasta n
        for i in range(2, n + 1):
            # Aplicamos la fórmula
            Hn = 2.0 * x * h_n_1 - 2.0 * (i - 1) * h_n_2

            # Actualizamos para la siguiente vuelta
            h_n_2 = h_n_1
            h_n_1 = Hn

    # --- 2. Gaussiana ---
    gauss = math.exp(- (t**2) / (2 * sigma**2))

    # --- 3. Constante normalizacion --
    K = 1.0 / math.sqrt(sigma * (2**n) * math.factorial(n) * math.sqrt(math.pi))

    return K * gauss * Hn

Repito código del 2.2 añadiendo nuevos parametros y en HALF Hermites

In [ ]:
# ---------- Parámetros ----------
THRESHOLD = 0.3
WAIT_SAMPLES = 72 # 200 ms a 360MHz
FS = 360
REFRACT_SAMPLES = 72
#no puede haber dos QRS muy seguidos
#dos picos casi jutnos en realidad forman parte del mismo latido
#dejamos un espacio llamado REFRACTORIO de mínimo 200ms en 360MHz => 72 samples

# ---------- Parámetros Hermites ----------
SIGMA = 5.0
NUM_COEFFS = 6
WINDOW = 36 # Mitad del latido (72/2)

# NUEVO ---------- Leer señal ----------
signal = record101.p_signal[:,0]   # canal 0
N = len(signal) #POR AHORA PARA PONER UN FINAL


# ---------- Inicializar variables ----------
state = "RESET"
last_confirmed = -100000
detected = []
K=4

# NUEVO ---------- Buffers ----------
tam_buf = 150 #dejamos margen
buf_latido = [0.0] * tam_buf
desciende = 0.0

# NUEVO ---------- Capturar Hermite ----------
resul_hermite = []

# NUEVO ---------- Analizamos Hermite ----------
def procesar_latido(fragmento_latido):
  # fragmento_latido es el array de 72 muestras
  # Copiamos
  datos_analizar = list(fragmento_latido) # (C -> memcpy o for)

  # Calculamos media
  suma = sum(datos_analizar)
  media = suma / len(datos_analizar)

  # Restamos media (centrar en 0)
  for j in range(len(datos_analizar)):
    datos_analizar[j] -= media

  # CALCULO COEFS
  mis_coefs = []
  for n in range(NUM_COEFFS):
    result = 0.0

    for j in range(len(datos_analizar)):
      t = j - WINDOW

      c_n = datos_analizar[j]

      phi_n = hermite(t, n, SIGMA)

      result += c_n * phi_n

    mis_coefs.append(result)

  return mis_coefs

# ---------- Bucle principal ----------
for i in range(K, N):

    # 0) BUFFER HERMITE
    buf_latido.pop(0) #borro el viejo
    buf_latido.append(signal[i]) #añado el nuevo


    # 1) derivada simple
    d = signal[i] - signal[i-k]
    v = abs(d)   # valor absoluto
    idx = i-K


    # ---------- Máquina de estados ----------
    if state == "RESET":
        # reiniciar
        state = "LOOKING"
        cand_val = -1.0 # valor candidato a QRS
        cand_idx = -1 # pos del candidato a QRS
        prov_best_idx = -1 # valor def QRS
        prov_best_val = -1.0 #pos de def QRS
        wait_until = -1


    if state == "LOOKING":
        if v > THRESHOLD and v > desciende: #si hay subida notable
            state = "PROV"
            cand_val = v
            cand_idx = idx


    elif state == "PROV":
        if v > cand_val: #si val > max anterior
            cand_val = v
            cand_idx = idx
        # si baja a menos de la mitad del pico provisional -> pasamos a HALF
        if v < 0.5 * cand_val:
            state = "HALF"
            prov_best_val = cand_val
            prov_best_idx = cand_idx
            wait_until = prov_best_idx + WAIT_SAMPLES #ESPERAMOS +72 samples


    elif state == "HALF":
        if v > prov_best_val: #si aparece v mayor que max hasta ahora
            state = "PROV" #volvemos a PROV
            cand_val = v
            cand_idx = idx
        else:
            # si esperamos suficiente, confirmamos el pico
            if i >= wait_until:
                if prov_best_idx - last_confirmed > REFRACT_SAMPLES:
                  if prov_best_val > desciende:
                    #si no cumple el REFRACTORIO no se confirma pico
                      detected.append(int(prov_best_idx))
                      last_confirmed = prov_best_idx #pos del ultimo pico detectado
                      desciende = prov_best_val

                      # NUEVO
                      # Calcular donde está el pico en el buf
                      lag = i - prov_best_idx
                      idx_buf = (tam_buf - 1) - lag

                      # Calcular ventana 72 muestras
                      inicio = idx_buf - WINDOW
                      fin = idx_buf + WINDOW

                      # Comprobamos que no nos salimos
                      if inicio >= 0 and fin <= tam_buf:
                          # Cortamos el trozo exacto de 72 muestras
                          recorte = buf_latido[inicio:fin]

                          # --- LLAMADA FUNCIÓN ---
                          coeficientes = procesar_latido(recorte)

                          # Guardar resultado
                          resul_hermite.append(coeficientes)

                      # reiniciar
                      state = "RESET"
    desciende = desciende * 0.99


# ---------- Mostrar resultados ----------
print(f"Total anotados por los médicos: {len(anotacion101.symbol)}")
print("Total de picos detectados:", len(detected))
print("Latidos analizados Hermites: ", len(resul_hermite))

In [ ]:
medical_annotations = list(anotacion101.sample)

detected_sorted = sorted(detected)
medical_annotations_sorted = sorted(medical_annotations)

print(f"First 20 detected peaks (sorted): {detected_sorted[:20]}")
print(f"First 20 medical annotations (sorted): {medical_annotations_sorted[:20]}")
print(f"Total detected peaks: {len(detected_sorted)}")
print(f"Total medical annotations: {len(medical_annotations_sorted)}")

In [ ]:
missed_peaks = []
matched_annotations = set()

for ann_idx in medical_annotations_sorted:
    found_match = False
    for det_idx in detected_sorted:
        if abs(ann_idx - det_idx) <= 6:
            found_match = True
            matched_annotations.add(ann_idx)
            break
    if not found_match:
        missed_peaks.append(ann_idx)

#-----------------------------------------------

false_positives = []
matched_detected = set()

for det_idx in detected_sorted:
    found_match = False
    for ann_idx in medical_annotations_sorted:
        if abs(det_idx - ann_idx) <= 6:
            found_match = True
            matched_detected.add(det_idx)
            break
    if not found_match:
        false_positives.append(det_idx)


print(f"\n--- Comparación resultados (Window: +/- 4 samples) ---")
print(f"Total annotaciones: {len(medical_annotations_sorted)}")
print(f"Total picos detectados: {len(detected_sorted)}")
print(f"Total picos perdidos: {len(missed_peaks)}")
print(f"Índices de picos perdidos: {missed_peaks}")
print(f"Total falsos positivos: {len(false_positives)}")
print(f"Indices falsos positivos: {false_positives}")

# Ver más

In [ ]:
# VISUALIZACIÓN DE MÚLTIPLES LATIDOS
import matplotlib.pyplot as plt

# Configuración
num_latidos_ver = 4

print(f"\nResultados finales:")
print(f"Total de picos detectados: {len(detected)}")
print(f"Latidos analizados Hermite: {len(resul_hermite)}")

if len(resul_hermite) > 0:

    # no pedir más latidos de los que hay
    cantidad_a_pintar = min(num_latidos_ver, len(resul_hermite))

    fig, axes = plt.subplots(cantidad_a_pintar, 1, figsize=(10, 3 * cantidad_a_pintar), sharex=True)

    if cantidad_a_pintar == 1:
        axes = [axes]

    t_grafica = np.arange(-WINDOW, WINDOW)


    for i in range(cantidad_a_pintar):

        # datos
        coefs = resul_hermite[i]
        pico_real_idx = detected[i]

        print(f"\nLatido num {i}")
        print(f"Ocurrió en la muestra global: {pico_real_idx}")
        print(f"Coeficientes calculados: {[round(c, 2) for c in coefs]}")

        # --- 1. Reconstrucción Matemática (Roja) ---
        reconstruccion = np.zeros_like(t_grafica, dtype=float)
        for n in range(NUM_COEFFS):
            molde = [hermite(tx, n, SIGMA) for tx in t_grafica]
            reconstruccion += np.array(molde) * coefs[n]

        # --- 2. Señal Real (Azul) ---
        inicio_real = pico_real_idx - WINDOW
        fin_real = pico_real_idx + WINDOW

        ax = axes[i]

        ax.plot(t_grafica, reconstruccion, 'r--', linewidth=2, label='Hermite (Modelo)')

        if inicio_real >= 0 and fin_real < len(signal):
            tramo_real = signal[inicio_real : fin_real]
            tramo_centrado = tramo_real - np.mean(tramo_real)

            ax.plot(t_grafica, tramo_centrado, 'b', alpha=0.6, label='Real (ECG)')
        else:
            ax.text(0, 0, "Latido en el borde (incompleto)", ha='center')

        ax.set_title(f"Latido {i} (Idx: {pico_real_idx}) | c0={coefs[0]:.2f}, c2={coefs[2]:.2f}")
        ax.grid(True)
        ax.set_ylabel("Amplitud")
        if i == 0:
            ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

else:
    print("No hay latidos procesados en 'resul_hermite'")

# El "Diccionario" de los Coeficientes Hermite

Los coeficientes describen la fisionomía del latido.
- c0 (Energía / Amplitud) Gauss:
  - Significado: dice "cuánta señal hay". Si el latido es muy alto y ancho, c0 será grande. Si es un latido débil, será pequeño.
  - Primer latido: 1.75. Es positivo, indica que la masa principal del latido va hacia arriba (como debe ser un QRS normal).
- c1 (Simetría / Tiempo) S:
  - Significado: dice si el latido está inclinado a la izquierda o la derecha.
  - Primer latido: -0.24.
- c2 (Anchura) W:
  - Significado: Mide la agudeza.
  - Primer latido: -1.5.
    - Un valor negativo fuerte significa que el latido es "estrecho y picudo" con bajadas a los lados (la forma típica Q-R-S sana).
    - Si este valor se acerca a 0 o se hace positivo, el latido es "gordo" o ventricular (peligroso).
- c3, c4, c5 (Detalles finos):
  - Significado: Son ondas con más "subidas y bajadas" (más frecuencia).
  - Sirven para capturar el ruido o detalles muy específicos de la forma del paciente.
  - Primer latido: 0.31, 0.2, 0.0. Son valores bajos. Eso es bueno, significa que el latido es "limpio" y se explica casi todo con los 3 primeros coeficientes. Si tuvieras mucho ruido eléctrico, estos valores subirían.
  
Resumen: La expansión de Hermite comprime la información. Los primeros coeficientes (c0, c1, c2) capturan la estructura general del complejo QRS (energía, posición y anchura), mientras que los siguientes 3 (c3, c4, c5) refinan los detalles de alta frecuencia.

In [ ]:
import numpy as np
import scipy.special as sp
import math

def hermite_functions(t, n, sigma):
    # La fórmula matemática para crear las formas
    norm = 1.0 / np.sqrt(sigma * (2**n) * math.factorial(n) * np.sqrt(np.pi))
    gauss = np.exp(-t**2 / (2 * sigma**2))
    Hn = sp.hermite(n)(t / sigma)
    return norm * gauss * Hn

linea_roja_hermite = np.full_like(signal, np.nan, dtype=float)

# Rellenar SÓLO los huecos donde hay latidos reconstruidos
for i, pico_idx in enumerate(detected):
    if i >= len(resul_hermite): break

    mis_nums = resul_hermite[i]

    inicio = int(pico_idx - WINDOW)
    fin = int(pico_idx + WINDOW)
    if inicio < 0 or fin >= len(signal): continue

    t_local = np.arange(-WINDOW, WINDOW)
    curva = np.zeros_like(t_local, dtype=float)

    for n in range(NUM_COEFFS):
        curva += mis_nums[n] * hermite_functions(t_local, n, SIGMA)

    nivel_base = np.mean(signal[inicio:fin])

    try:
        linea_roja_hermite[inicio:fin] = curva + nivel_base
    except ValueError: pass

print("Línea roja calculada")

In [ ]:
X = 0
ANCHO = 20

t = np.arange(len(signal)) / FS

plt.figure(figsize=(16, 7))
plt.plot(t, signal, color='steelblue', alpha=0.6, label='ECG Real (db)', linewidth=0.8)
plt.plot(t, linea_roja_hermite, color='red', linestyle='--', linewidth=1.5, label='Reconstrucción Hermite')

detected_valid_indices = [idx for idx in detected if idx < len(signal)]
plt.scatter(np.array(detected_valid_indices) / FS, signal[detected_valid_indices],
            color='red', marker='o', s=50, label='Picos Detectados (Algoritmo)', zorder=5)

anotacion101_valid_samples = [s for s in anotacion101.sample if s < len(signal)]
plt.scatter(np.array(anotacion101_valid_samples) / FS, signal[anotacion101_valid_samples],
            color='green', marker='x', s=80, label='Anotaciones Médicas', zorder=6)
plt.title('ECG Signal, Hermite Reconstruction, Detected Peaks, and Medical Annotations')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.xlim(X, X + ANCHO)
plt.ylim(-1, 1.5)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
index = 11384
approx_time_seconds = index / FS
print(f"El índice {index} está aproximadamente en el segundo {approx_time_seconds:.2f}.")

In [ ]:
# ---------- Parámetros ----------
THRESHOLD = 0.3
WAIT_SAMPLES = 72 #200 ms a 360MHz
FS = 360
REFRACT_SAMPLES = 72
#no puede haber dos QRS muy seguidos
#dos picos casi jutnos en realidad forman parte del mismo latido
#dejamos un espacio llamado REFRACTORIO de mínimo 200ms en 360MHz => 72 samples

# NUEVO ---------- Parámetros Hermites ----------
SIGMA = 5.0
NUM_COEFFS = 6
WINDOW = 36 # Mitad del latido (72/2)

# ---------- Leer señal ----------
signal = record101.p_signal[:,0]   # canal 0
N = len(signal) #POR AHORA PARA PONER UN FINAL


# ---------- Inicializar variables ----------
state = "RESET"
last_confirmed = -100000
detected = []
k=3

# NUEVO ---------- Buffers ----------
tam_buf = 150 #dejamos margen
buf_latido = [0.0] * tam_buf
desciende = 0.0
desciende_history = []

# NUEVO ---------- Capturar Hermite ----------
resul_hermite = []

# NUEVO ---------- Analizamos Hermite ----------
def procesar_latido(fragmento_latido):
  # fragmento_latido es el array de 72 muestras
  # Copiamos
  datos_analizar = list(fragmento_latido) # (C -> memcpy o for)

  # Calculamos media
  suma = sum(datos_analizar)
  media = suma / len(datos_analizar)

  # Restamos media (centrar en 0)
  for j in range(len(datos_analizar)):
    datos_analizar[j] -= media

  # CALCULO COEFS
  mis_coefs = []
  for n in range(NUM_COEFFS):
    result = 0.0

    for j in range(len(datos_analizar)):
      t = j - WINDOW

      c_n = datos_analizar[j]

      phi_n = hermite(t, n, SIGMA)

      result += c_n * phi_n

    mis_coefs.append(result)

  return mis_coefs

# ---------- Bucle principal ----------
for i in range(k, N):

    # 0) BUFFER HERMITE
    buf_latido.pop(0) #borro el viejo
    buf_latido.append(signal[i]) #añado el nuevo


    # 1) derivada simple
    d = signal[i] - signal[i-k]
    v = abs(d)   # valor absoluto
    idx = i-k

    desciende = desciende * 0.99

    # ---------- Máquina de estados ----------
    if state == "RESET":
        # reiniciar
        state = "LOOKING"
        cand_val = -1.0 # valor candidato a QRS
        cand_idx = -1 # pos del candidato a QRS
        prov_best_idx = -1 # valor def QRS
        prov_best_val = -1.0 #pos de def QRS
        wait_until = -1


    if state == "LOOKING":
        if v > THRESHOLD and v > desciende: #si hay subida notable
            state = "PROV"
            cand_val = v
            cand_idx = idx


    elif state == "PROV":
        if v > cand_val: #si val > max anterior
            cand_val = v
            cand_idx = idx
        # si baja a menos de la mitad del pico provisional -> pasamos a HALF
        if v < 0.5 * cand_val:
            state = "HALF"
            prov_best_val = cand_val
            prov_best_idx = cand_idx
            wait_until = prov_best_idx + WAIT_SAMPLES #ESPERAMOS +72 samples


    elif state == "HALF":
        if v > prov_best_val and v > desciende: #si aparece v mayor que max hasta ahora
            state = "PROV" #volvemos a PROV
            cand_val = v
            cand_idx = idx
        else:
            # si esperamos suficiente, confirmamos el pico
            if i >= wait_until:
                if prov_best_idx - last_confirmed > REFRACT_SAMPLES:
                  #if cand_val > desciende:
                #si no cumple el REFRACTORIO no se confirma pico
                  detected.append(int(prov_best_idx))
                  last_confirmed = prov_best_idx #pos del ultimo pico detectado
                  desciende = prov_best_val

                  # NUEVO
                  # Calcular donde está el pico en el buf
                  lag = i - prov_best_idx
                  idx_buf = (tam_buf - 1) - lag

                  # Calcular ventana 72 muestras
                  inicio = idx_buf - WINDOW
                  fin = idx_buf + WINDOW

                  # Comprobar que no no se sale
                  if inicio >= 0 and fin <= tam_buf:
                      # Cortamos el trozo exacto de 72 muestras
                      recorte = buf_latido[inicio:fin]

                      # --- LLAMADA FUNCIÓN ---
                      coeficientes = procesar_latido(recorte)

                      # Guardar resultado
                      resul_hermite.append(coeficientes)

                  # reiniciar
                  state = "RESET"
    desciende_history.append(desciende)


# ---------- resultados ----------
print(f"Total anotados por los médicos: {len(anotacion101.symbol)}")
print("Total de picos detectados:", len(detected))
print("Latidos analizados Hermites: ", len(resul_hermite))
print("Longitud del historial de desciende:", len(desciende_history))

if len(resul_hermite) > 0:
  latido_idx = 0 # primer latido
  coefs = resul_hermite[latido_idx]
  pico_real_idx = detected[latido_idx] # Índice donde ocurrió el pic

  print(f"Pintando latido número {latido_idx}")
  print(f"Ocurrió en la muestra: {pico_real_idx}")
  print(f"Coeficientes calculados: {[round(c, 2) for c in coefs]}")

  # Generar la curva roja usando los coefs
  t_grafica = np.arange(-WINDOW, WINDOW)
  reconstruccion = np.zeros_like(t_grafica, dtype=float)

  for n in range(NUM_COEFFS):
      molde = [hermite(tx, n, SIGMA) for tx in t_grafica]
      reconstruccion += np.array(molde) * coefs[n]

  plt.figure(figsize=(10,6))
  plt.title(f"Reconstrucción Hermite (Latido en muestra {pico_real_idx})")

  # 1. Línea roja
  plt.plot(t_grafica, reconstruccion, 'r--', label='Hermite')

  # 2. Línea azul (datos reales)
  inicio_real = pico_real_idx - WINDOW
  fin_real = pico_real_idx + WINDOW
  if inicio_real >= 0 and fin_real < len(signal):
      tramo_real = signal[inicio_real : fin_real]
      plt.plot(t_grafica, tramo_real - np.mean(tramo_real), 'b', alpha=0.5, label='Real')
  else:
      print("El latido real está en el borde y no se puede pintar entero")

  plt.legend()
  plt.grid()
  plt.show()

else:
  print("No hay suficientes latidos")

In [ ]:
X = 0
ANCHO = 10

# Eje de tiempo
t = np.arange(len(signal)) / FS

plt.figure(figsize=(16, 7))
plt.plot(t, signal, color='steelblue', alpha=0.6, label='ECG Real (Database)', linewidth=0.8)
plt.plot(t, linea_roja_hermite, color='red', linestyle='--', linewidth=1.5, label='Hermite Reconstrucción')

detected_valid_indices = [idx for idx in detected if idx < len(signal)]
plt.scatter(np.array(detected_valid_indices) / FS, signal[detected_valid_indices],
            color='red', marker='o', s=50, label='Picos Detectados (Algoritmo)', zorder=5)

anotacion101_valid_samples = [s for s in anotacion101.sample if s < len(signal)]
plt.scatter(np.array(anotacion101_valid_samples) / FS, signal[anotacion101_valid_samples],
            color='green', marker='x', s=80, label='Anotaciones Médicas', zorder=6)

if len(desciende_history) > 0:
    time_desciende = np.arange(k, k + len(desciende_history)) / FS
    plt.plot(time_desciende, desciende_history, color='orange', linestyle='-', linewidth=1, label='Desciende History')


plt.title('ECG Signal, Hermite Reconstruction, Detected Peaks, Medical Annotations, and Desciende History')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.xlim(X, X + ANCHO)
plt.ylim(-1, 1.5)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()